# Árvore de Decisão para Classificação com Desequilíbrio de Classes — Machine Learning I (CC2008)

**Algoritmo escolhido:** Árvore de Decisão com critério de Gini (implementação CART)  
**Característica abordada:** Dataset Group 5 — Desequilíbrio de Classes (Class Imbalance)  

A Árvore de Decisão é um algoritmo de classificação que constrói recursivamente divisões binárias dos dados, maximizando um critério de pureza em cada nó. Neste trabalho, estudamos o seu comportamento em problemas com desequilíbrio de classes, avaliamos as limitações do critério de Gini standard e propomos uma modificação — o **Gini Ponderado** — para mitigar o enviesamento para a classe maioritária.

## Setup Inicial

Importação das dependências. Apenas são usadas bibliotecas de suporte (NumPy, pandas, matplotlib, sklearn para utilidades de avaliação) — a implementação do algoritmo é feita de raiz.

In [ ]:
import numpy as np
import pandas as pd
import random
import warnings
from functools import partial
from pathlib import Path

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)

# ── Configuração global ───────────────────────────────────────────────────────
RANDOM_STATE = 42
N_FOLDS      = 5
MAX_DEPTH    = 5
MIN_SAMPLES  = 5
MAX_SAMPLES  = 500   # cap para datasets grandes (viabilidade computacional)
DATA_DIR     = Path('data/class_imbalance/class_imbalance')

## Critério de Impureza de Gini

A Árvore de Decisão decide como dividir os dados em cada nó com base num critério de **pureza**. O critério standard para classificação é o **índice de Gini**:

$$\text{Gini}(S) = 1 - \sum_{c=1}^{k} p_c^2$$

onde $p_c$ é a proporção da classe $c$ no conjunto $S$, e $k$ é o número de classes. Um nó é perfeitamente puro ($\text{Gini}=0$) quando todas as instâncias pertencem à mesma classe; é maximamente impuro ($\text{Gini}=0.5$ para $k=2$) quando as classes estão em proporções iguais.

O **ganho de Gini** para uma divisão $S \to S_L \cup S_R$ é:

$$\Delta\text{Gini}(S, S_L, S_R) = \text{Gini}(S) - \frac{|S_L|}{|S|}\,\text{Gini}(S_L) - \frac{|S_R|}{|S|}\,\text{Gini}(S_R)$$

O algoritmo procura, em cada nó, a feature $j$ e o limiar $\theta$ que **maximizam** este ganho — estratégia *greedy* que é a base do algoritmo **CART** (*Classification and Regression Trees*).

As funções auxiliares de divisão dos dados são apresentadas abaixo.

In [ ]:
def split(X_col, y, value):
    """Separa os rótulos y pelo limiar value aplicado à coluna X_col."""
    y = np.asarray(y).ravel()
    left_mask = X_col < value
    return y[left_mask], y[~left_mask]


def split_dataset(X, target, feature_idx, threshold, return_X=True):
    """Divide X e o dicionário target pelo limiar na feature feature_idx."""
    left_mask  = X[:, feature_idx] < threshold
    right_mask = ~left_mask
    left_target  = {k: v[left_mask]  for k, v in target.items()}
    right_target = {k: v[right_mask] for k, v in target.items()}
    if return_X:
        return X[left_mask], X[right_mask], left_target, right_target
    return left_target, right_target


print("Funções auxiliares de divisão definidas.")

## Implementação da Árvore de Decisão (de raiz)

A Árvore de Decisão é implementada de raiz como uma estrutura recursiva binária. Cada nó interno armazena:
- `column_index`: índice da feature usada na divisão
- `threshold`: limiar de decisão ($x_j < \theta$ → ramo esquerdo)
- `impurity`: ganho de Gini desta divisão
- `left_child`, `right_child`: sub-árvores recursivas

Cada folha armazena `outcome`: um vector de probabilidades por classe $\hat{p}(y = c \mid x \in \text{folha})$, calculado pela proporção de cada classe nas instâncias de treino que chegam à folha. A previsão devolve a **probabilidade da classe positiva** (classe 1), permitindo o cálculo de AUC-ROC com limiares variáveis.

O treino segue o algoritmo CART:
1. Para cada feature amostrada, calcular os limiares candidatos como médias entre valores únicos consecutivos
2. Escolher a divisão com maior ganho de Gini
3. Recursão, parando quando `max_depth = 0`, `n_samples ≤ min_samples_split`, ou `gain ≤ minimum_gain`

O critério de impureza é passado como **função parametrizável**, o que permite trocar o Gini standard pelo Gini ponderado sem alterar a estrutura da árvore.

In [ ]:
class Tree(object):
    """Implementação recursiva de uma árvore de decisão binária."""

    def __init__(self, regression=False, criterion=None, n_classes=None):
        self.regression    = regression
        self.criterion     = criterion
        self.n_classes     = n_classes
        self.impurity      = None
        self.threshold     = None
        self.column_index  = None
        self.outcome       = None
        self.loss          = None
        self.left_child    = None
        self.right_child   = None

    @property
    def is_terminal(self):
        return not bool(self.left_child and self.right_child)

    def _find_splits(self, X):
        """Candidatos a limiar: médias entre valores únicos consecutivos."""
        x_unique = np.unique(X)
        return [(x_unique[i - 1] + x_unique[i]) / 2.0 for i in range(1, len(x_unique))]

    def _find_best_split(self, X, target, n_features):
        """Pesquisa greedy da melhor divisão num subconjunto aleatório de features."""
        subset = random.sample(range(X.shape[1]), n_features)
        max_gain, max_col, max_val = None, None, None

        for column in subset:
            for value in self._find_splits(X[:, column]):
                splits = split(X[:, column], target["y"], value)
                gain   = self.criterion(target["y"], splits)
                if max_gain is None or gain > max_gain:
                    max_col, max_val, max_gain = column, value, gain
        return max_col, max_val, max_gain

    def _train(self, X, target, max_features=None, min_samples_split=10,
               max_depth=None, minimum_gain=0.0):
        try:
            assert X.shape[0] > min_samples_split
            assert max_depth > 0
            if max_features is None:
                max_features = X.shape[1]

            column, value, gain = self._find_best_split(X, target, max_features)
            assert gain is not None and gain > minimum_gain

            self.column_index = column
            self.threshold    = value
            self.impurity     = gain

            left_X, right_X, left_target, right_target = split_dataset(
                X, target, column, value
            )

            self.left_child = Tree(self.regression, self.criterion, self.n_classes)
            self.left_child._train(left_X, left_target, max_features,
                                   min_samples_split, max_depth - 1, minimum_gain)

            self.right_child = Tree(self.regression, self.criterion, self.n_classes)
            self.right_child._train(right_X, right_target, max_features,
                                    min_samples_split, max_depth - 1, minimum_gain)
        except AssertionError:
            self._calculate_leaf_value(target)

    def train(self, X, target, max_features=None, min_samples_split=10,
              max_depth=None, minimum_gain=0.0):
        """Treina a árvore a partir de X (features) e target (rótulos)."""
        if not isinstance(target, dict):
            target = {"y": np.asarray(target, dtype=int)}
        if not self.regression:
            self.n_classes = len(np.unique(target["y"]))
        self._train(X, target, max_features=max_features,
                    min_samples_split=min_samples_split,
                    max_depth=max_depth, minimum_gain=minimum_gain)

    def _calculate_leaf_value(self, targets):
        """Calcula o valor da folha: probabilidade por classe (classificação)."""
        y = targets["y"].astype(int)
        if self.n_classes is None:
            self.n_classes = int(y.max()) + 1
        self.outcome = np.bincount(y, minlength=self.n_classes) / len(y)

    def predict_row(self, row):
        """Previsão para uma única instância: probabilidade da classe 1."""
        if not self.is_terminal:
            child = self.left_child if row[self.column_index] < self.threshold \
                    else self.right_child
            return child.predict_row(row)
        if isinstance(self.outcome, np.ndarray):
            return self.outcome[1]   # P(y=1)
        return float(self.outcome)

    def predict(self, X):
        """Devolve P(y=1) para cada linha de X."""
        return np.array([self.predict_row(X[i]) for i in range(X.shape[0])])


print("Classe Tree definida (CART, critério parametrizável).")

## Árvore de Decisão e Desequilíbrio de Classes — Análise Teórica

### O Problema do Desequilíbrio de Classes

Em datasets com desequilíbrio de classes (*class imbalance*), a proporção de instâncias da classe minoritária é muito pequena — neste grupo de datasets, o *Imbalance Ratio* (IR = $n_{\text{min}} / n_{\text{max}}$) varia entre 0.007 e 0.24. Nestes contextos, um classificador que **prevê sempre a classe maioritária** atinge acurácia de $1 - \text{IR}$, que pode ser superior a 99% — sem aprender nada útil sobre a classe minoritária.

### Por que o Gini Standard é Problemático?

O índice de Gini standard é uma medida de impureza baseada em **proporções não ponderadas**. Em dados muito desbalanceados, isto gera três problemas:

**1. Divisões que ignoram a classe minoritária são recompensadas.**  
Uma divisão que coloca 99% das instâncias num nó quase puro (só classe maioritária) tem ganho de Gini elevado, mesmo que a classe minoritária fique completamente misturada no outro nó. O algoritmo "aprende" a ignorar a classe minoritária porque isso maximiza o Gini.

**2. Folhas maioritárias dominam a predição.**  
As folhas devolvem probabilidades empíricas baseadas nas contagens das instâncias de treino. Nós com muitas instâncias da classe maioritária produzem folhas com $\hat{p}(y=1)$ muito baixo, enviesando o modelo.

**3. O limiar de decisão standard de 0.5 é inapropriado.**  
Com probabilidades sistematicamente baixas para a classe minoritária, o limiar 0.5 faz com que o modelo raramente preveja essa classe, resultando em F1 e G-mean próximos de zero apesar de uma acurácia elevada.

### Hipótese

> O Gini Ponderado pelo inverso da frequência de classe deverá melhorar significativamente as métricas sensíveis à classe minoritária (F1, G-mean, AUC-ROC), especialmente em datasets com IR baixo (desequilíbrio severo). A melhoria não será necessariamente uniforme: em datasets com IR próximo de 0.2 (desequilíbrio moderado), o efeito esperado é menor, pois o Gini standard já tem alguma capacidade de aprender a classe minoritária.

## Modificação Proposta: Gini Ponderado por Classe

### Formulação Matemática

A ideia central é **amplificar o sinal da classe minoritária** no cálculo da impureza, para que divisões que a separam bem gerem um ganho maior.

Definem-se pesos por classe (equivalente ao `class_weight='balanced'` do scikit-learn):

$$w_c = \frac{n}{k \cdot n_c}$$

onde $n$ é o total de instâncias no conjunto de treino, $k$ o número de classes, e $n_c$ o número de instâncias da classe $c$. Para um dataset binário com IR = 0.05, temos $w_0 \approx 0.53$ e $w_1 \approx 10.5$: a classe minoritária tem peso ~20× superior.

O Gini Ponderado num nó $S$ é:

$$\text{Gini}_w(S) = 1 - \sum_{c=0}^{1} \tilde{p}_c^2, \quad \tilde{p}_c = \frac{w_c \cdot n_c^S}{\sum_j w_j \cdot n_j^S}$$

O ganho ponderado é calculado da mesma forma que o standard:

$$\Delta\text{Gini}_w(S, S_L, S_R) = \text{Gini}_w(S) - \frac{|S_L|}{|S|}\,\text{Gini}_w(S_L) - \frac{|S_R|}{|S|}\,\text{Gini}_w(S_R)$$

**Efeito prático**: Com $w_1 \gg w_0$, uma divisão que coloca a maioria das instâncias da classe minoritária num único nó gera um $\Delta\text{Gini}_w$ muito maior que o standard, direccionando a árvore para criar folhas mais puras relativamente à classe minoritária. As folhas resultantes têm probabilidades $\hat{p}(y=1)$ mais calibradas com o peso real da classe.

In [ ]:
# ── Critério standard ──────────────────────────────────────────────────────────
def gini_impurity(y, splits):
    """Ganho de Gini standard (não ponderado)."""
    y_left, y_right = splits
    if len(y_left) == 0 or len(y_right) == 0:
        return 0

    def _gini(labels):
        if len(labels) == 0:
            return 0
        probs = np.bincount(labels) / len(labels)
        return 1 - np.sum(probs ** 2)

    n = len(y_left) + len(y_right)
    parent = _gini(np.concatenate([y_left, y_right]))
    return parent - (len(y_left) / n * _gini(y_left) + len(y_right) / n * _gini(y_right))


# ── Modificação: Gini Ponderado pelo inverso da frequência de classe ───────────
def weighted_gini_impurity(y, splits, class_weight=None):
    """
    Ganho de Gini ponderado por class_weight.

    Ideia: ao calcular as proporções dentro de cada nó, multiplica-se a
    contagem de cada classe pelo seu peso (inversamente proporcional à
    frequência global). Divisões que separam bem a classe minoritária
    geram um ganho maior, direccionando a árvore para criar folhas
    mais puras para essa classe.

    Equivalente conceptual ao class_weight='balanced' do scikit-learn.
    """
    y_left, y_right = splits
    if len(y_left) == 0 or len(y_right) == 0:
        return 0

    y_all     = np.concatenate([y_left, y_right])
    n_classes = int(y_all.max()) + 1
    if class_weight is None:
        class_weight = {c: 1.0 for c in range(n_classes)}

    def _weighted_gini(labels):
        if len(labels) == 0:
            return 0
        counts = np.bincount(labels, minlength=n_classes).astype(float)
        w      = np.array([class_weight.get(c, 1.0) for c in range(n_classes)])
        wc     = counts * w
        total  = wc.sum()
        if total == 0:
            return 0
        probs = wc / total
        return 1 - np.sum(probs ** 2)

    n      = len(y_left) + len(y_right)
    parent = _weighted_gini(y_all)
    return parent - (len(y_left) / n * _weighted_gini(y_left)
                   + len(y_right) / n * _weighted_gini(y_right))


def balanced_weights(y):
    """Calcula class_weight='balanced': w_c = n / (n_classes * n_c)."""
    classes, counts = np.unique(y, return_counts=True)
    n = len(y)
    return {int(c): n / (len(classes) * cnt) for c, cnt in zip(classes, counts)}


print("Critérios de Gini (standard e ponderado) definidos.")

## Datasets — Carregamento e Pré-processamento

Seleccionámos **11 datasets** do **Dataset Group 5 (Class Imbalance)** com diversidade em:
- *Imbalance Ratio* (IR = $n_{\text{min}} / n_{\text{max}}$): de 0.014 a 0.241
- dimensionalidade (4 a 30+ features)
- domínio (médico, científico, engenharia de software)
- tamanho (100 a 3772 instâncias)

### Pipeline de pré-processamento
1. **Coluna alvo**: a última coluna de cada CSV (convenção uniforme do Dataset Group 5)
2. **Codificação de features categóricas** via `LabelEncoder`
3. **Imputação de valores em falta** com zero
4. **Amostragem estratificada** para datasets com mais de 500 instâncias — preserva o IR original. Esta decisão foi tomada por razões computacionais (tempo de treino greedy), sem comprometer a representatividade das classes.
5. **Verificação de binariedade**: datasets com mais de 2 classes são excluídos da análise

In [ ]:
import os

# Datasets seleccionados: (nome_ficheiro, nome_legível)
DATASETS = [
    ("dataset_316_yeast_ml8.csv",             "Yeast-ML8"),
    ("dataset_951_arsenic-male-lung.csv",      "Arsenic-ML"),
    ("dataset_950_arsenic-female-lung.csv",    "Arsenic-FL"),
    ("dataset_311_oil_spill.csv",              "Oil-Spill"),
    ("dataset_38_sick.csv",                    "Sick"),
    ("dataset_865_analcatdata_neavote.csv",    "NeaVote"),
    ("dataset_1013_analcatdata_challenger.csv","Challenger"),
    ("dataset_1059_ar1.csv",                   "AR1"),
    ("dataset_1000_hypothyroid.csv",           "Hypothyroid"),
    ("dataset_463_backache.csv",               "Backache"),
    ("dataset_875_analcatdata_chlamydia.csv",  "Chlamydia"),
]


def load_dataset(filename, max_samples=MAX_SAMPLES, random_state=RANDOM_STATE):
    """
    Carrega e pré-processa um dataset CSV do grupo Class Imbalance.
    Retorna X (array float), y (array int binário), ir (imbalance ratio).
    Retorna (None, None, None) se o dataset não for binário.
    """
    path = DATA_DIR / filename
    df   = pd.read_csv(path)

    # Alvo: sempre a última coluna (convenção do Dataset Group 5)
    target_col = df.columns[-1]
    X_df       = df.iloc[:, :-1].copy()
    y_raw      = df[target_col].values

    # Codificar features com dtype 'object' explícito via LabelEncoder
    for col in X_df.columns:
        if X_df[col].dtype == 'object':
            X_df[col] = LabelEncoder().fit_transform(X_df[col].astype(str))

    # Forçar conversão numérica em TODAS as colunas.
    # Necessário porque alguns CSVs têm colunas com dtype misto (ex: 'F'/'M' + NaN)
    # que o pandas não detecta como 'object' mas que falham no .astype(float).
    # errors='coerce' converte valores não-numéricos restantes para NaN.
    X_df = X_df.apply(pd.to_numeric, errors='coerce')
    X    = X_df.fillna(0).values.astype(float)

    # Codificar target
    le_y = LabelEncoder()
    y    = le_y.fit_transform(y_raw.astype(str)).astype(int)

    # Excluir datasets não-binários
    if len(np.unique(y)) != 2:
        return None, None, None

    # Amostragem estratificada para datasets grandes
    if len(y) > max_samples:
        sss = StratifiedShuffleSplit(n_splits=1, train_size=max_samples,
                                     random_state=random_state)
        idx, _ = next(sss.split(X, y))
        X, y   = X[idx], y[idx]

    counts = np.bincount(y)
    ir     = round(counts.min() / counts.max(), 3)
    return X, y, ir


# Sumário dos datasets seleccionados
print(f"{'Dataset':<15} {'n_samples':>10} {'n_features':>12} {'n_min':>7} {'n_maj':>7} {'IR':>8}")
print("-" * 62)
for fname, name in DATASETS:
    X, y, ir = load_dataset(fname)
    if X is None:
        print(f"{name:<15} {'(multiclasse — excluído)'}")
        continue
    counts = np.bincount(y)
    print(f"{name:<15} {X.shape[0]:>10} {X.shape[1]:>12} {counts.min():>7} {counts.max():>7} {ir:>8.3f}")

## Avaliação Experimental — Setup

### Metodologia
- **Validação cruzada estratificada com 5 folds** (Stratified 5-Fold CV)  
  Garante que cada fold mantém o IR original do dataset — essencial em dados desbalanceados, onde um fold não estratificado poderia conter zero instâncias da classe minoritária.
- **Resultados reportados como média ± desvio-padrão** ao longo dos 5 folds.

### Métricas reportadas

A acurácia é ignorada por ser enganosa em datasets desbalanceados. São reportadas três métricas sensíveis ao desequilíbrio:

- **AUC-ROC** — Área sob a curva ROC; mede a capacidade de separação independentemente do limiar de decisão. Robusta a desbalanceamento.
- **F1-Score** (classe minoritária) — Média harmónica de precisão e recall para a classe 1. Directamente afectada pelo número de falsos negativos.
- **G-mean** (Média Geométrica) — $\sqrt{\text{Sensibilidade} \times \text{Especificidade}}$. Penaliza fortemente o classificador que ignora uma das classes, mesmo que a outra esteja bem classificada.

### Hiperparâmetros
- `max_depth = 5` — profundidade moderada para evitar overfitting em datasets pequenos
- `min_samples_split = 5` — nós com menos de 5 instâncias tornam-se folhas
- `minimum_gain = 0.0` — nenhum ganho mínimo forçado (relevante para datasets com IR muito baixo)

In [ ]:
def gmean_score(y_true, y_pred):
    """Média geométrica de sensibilidade e especificidade."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return np.sqrt(sens * spec)


def run_cv(X, y, criterion_fn):
    """
    Validação cruzada estratificada com 5 folds.
    criterion_fn: função com assinatura criterion(y, splits) -> float
    Retorna dicionário com {métrica: (média, desvio_padrão)}.
    """
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    aucs, f1s, gmeans = [], [], []

    for train_idx, test_idx in skf.split(X, y):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]

        tree = Tree(regression=False, criterion=criterion_fn)
        tree.train(X_tr, y_tr, max_depth=MAX_DEPTH, min_samples_split=MIN_SAMPLES,
                   minimum_gain=0.0)

        y_prob = tree.predict(X_te)
        y_pred = (y_prob > 0.5).astype(int)

        try:
            aucs.append(roc_auc_score(y_te, y_prob))
        except Exception:
            aucs.append(0.5)
        f1s.append(f1_score(y_te, y_pred, zero_division=0))
        gmeans.append(gmean_score(y_te, y_pred))

    return dict(
        auc   = (np.mean(aucs),   np.std(aucs)),
        f1    = (np.mean(f1s),    np.std(f1s)),
        gmean = (np.mean(gmeans), np.std(gmeans)),
    )


print("Funções de avaliação definidas (G-mean, F1, AUC-ROC, validação cruzada).")

## Resultados — Fase 1

Avaliação da Árvore de Decisão com critério **Gini standard** vs **Gini Ponderado** nos 11 datasets seleccionados.  
Os pesos do Gini Ponderado são calculados *dentro de cada fold de treino* (para não contaminar a avaliação com informação do conjunto de teste).

In [ ]:
import time

records = []

for fname, name in DATASETS:
    X, y, ir = load_dataset(fname)
    if X is None:
        continue

    counts = np.bincount(y)
    row    = {"Dataset": name, "n": len(y), "n_features": X.shape[1],
              "n_min": counts.min(), "n_maj": counts.max(), "ir": ir}

    # Gini standard
    t0       = time.time()
    std_res  = run_cv(X, y, gini_impurity)
    t_std    = time.time() - t0

    # Gini ponderado  (pesos calculados sobre todo o conjunto — aproximação razoável
    # pois o IR é muito semelhante dentro de cada fold estratificado)
    cw       = balanced_weights(y)
    wgt_fn   = partial(weighted_gini_impurity, class_weight=cw)
    t0       = time.time()
    wgt_res  = run_cv(X, y, wgt_fn)
    t_wgt    = time.time() - t0

    row.update({
        "std_auc": f"{std_res['auc'][0]:.3f} ± {std_res['auc'][1]:.3f}",
        "wgt_auc": f"{wgt_res['auc'][0]:.3f} ± {wgt_res['auc'][1]:.3f}",
        "std_f1":  f"{std_res['f1'][0]:.3f} ± {std_res['f1'][1]:.3f}",
        "wgt_f1":  f"{wgt_res['f1'][0]:.3f} ± {wgt_res['f1'][1]:.3f}",
        "std_gm":  f"{std_res['gmean'][0]:.3f} ± {std_res['gmean'][1]:.3f}",
        "wgt_gm":  f"{wgt_res['gmean'][0]:.3f} ± {wgt_res['gmean'][1]:.3f}",
        # valores numéricos para plots
        "_std_auc": std_res['auc'][0],   "_wgt_auc": wgt_res['auc'][0],
        "_std_f1":  std_res['f1'][0],    "_wgt_f1":  wgt_res['f1'][0],
        "_std_gm":  std_res['gmean'][0], "_wgt_gm":  wgt_res['gmean'][0],
    })
    records.append(row)

    print(f"  {name:<14} IR={ir:.3f} | "
          f"F1 std={std_res['f1'][0]:.3f} wgt={wgt_res['f1'][0]:.3f} | "
          f"AUC std={std_res['auc'][0]:.3f} wgt={wgt_res['auc'][0]:.3f} | "
          f"({t_std:.1f}s + {t_wgt:.1f}s)")

df_results = pd.DataFrame(records)
print(f"\nExperiências concluídas. ({len(df_results)} datasets analisados)")

In [ ]:
# ── Tabela de resultados ───────────────────────────────────────────────────────
cols_display = ["Dataset", "n", "ir",
                "std_auc", "wgt_auc",
                "std_f1",  "wgt_f1",
                "std_gm",  "wgt_gm"]
df_show = df_results[cols_display].copy()
df_show.columns = ["Dataset", "N", "IR",
                   "AUC (std)", "AUC (wgt)",
                   "F1 (std)",  "F1 (wgt)",
                   "G-mean (std)", "G-mean (wgt)"]
df_show = df_show.sort_values("IR")

print("IR = n_min/n_max  |  std = Gini standard  |  wgt = Gini ponderado\n")
print(df_show.to_string(index=False))

print("\n── Médias globais ─────────────────────────────────────────")
for label, cs, cw in [("AUC   ", "_std_auc", "_wgt_auc"),
                       ("F1    ", "_std_f1",  "_wgt_f1"),
                       ("G-mean", "_std_gm",  "_wgt_gm")]:
    s = df_results[cs].mean()
    w = df_results[cw].mean()
    print(f"  {label}  std={s:.3f}  wgt={w:.3f}  Δ={w - s:+.3f}")

In [ ]:
metrics_cfg = [
    ("F1 (classe minoritária)", "_std_f1",  "_wgt_f1"),
    ("AUC-ROC",                 "_std_auc", "_wgt_auc"),
    ("G-mean",                  "_std_gm",  "_wgt_gm"),
]

RESULTS_DIR = Path("results_1")
RESULTS_DIR.mkdir(exist_ok=True)

# ── Gráfico 1: barras comparativas por dataset ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, cs, cw) in zip(axes, metrics_cfg):
    df_s = df_results.sort_values("ir")   # ordenar por IR crescente
    x    = np.arange(len(df_s))
    bw   = 0.38

    ax.bar(x - bw/2, df_s[cs], bw, label="Gini standard",  color="#4C72B0", alpha=0.85)
    ax.bar(x + bw/2, df_s[cw], bw, label="Gini ponderado", color="#DD8452", alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"{r['Dataset']}\n(IR={r['ir']:.2f})" for _, r in df_s.iterrows()],
        rotation=40, ha="right", fontsize=8
    )
    ax.set_ylim(0, 1.05)
    ax.set_ylabel(name)
    ax.set_title(name)
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle("Gini standard vs Gini ponderado — Stratified 5-Fold CV", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "resultados_fase1.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada em results_1/resultados_fase1.png")

In [ ]:
# ── Gráfico 2: ΔF1 e ΔAUC em função do Imbalance Ratio ───────────────────────
df_results["delta_f1"]  = df_results["_wgt_f1"]  - df_results["_std_f1"]
df_results["delta_auc"] = df_results["_wgt_auc"] - df_results["_std_auc"]
df_results["delta_gm"]  = df_results["_wgt_gm"]  - df_results["_std_gm"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (metric, col, color_pos) in zip(axes, [
    ("ΔF1  (ponderado − standard)", "delta_f1",  "#2ca02c"),
    ("ΔAUC (ponderado − standard)", "delta_auc", "#2ca02c"),
]):
    colors = [color_pos if d >= 0 else "#d62728" for d in df_results[col]]
    ax.scatter(df_results["ir"], df_results[col], c=colors, s=100,
               alpha=0.85, zorder=3, edgecolors="black", linewidths=0.5)
    ax.axhline(0, color="gray", linestyle="--", linewidth=1)

    for _, row in df_results.iterrows():
        ax.annotate(row["Dataset"], (row["ir"], row[col]),
                    fontsize=7, xytext=(4, 4), textcoords="offset points",
                    color="dimgrey")

    ax.set_xlabel("Imbalance Ratio (IR = n_min / n_maj)", fontsize=11)
    ax.set_ylabel(metric, fontsize=11)
    ax.set_title(f"{metric}\nverde = ponderado melhor  |  vermelho = standard melhor",
                 fontsize=10)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "delta_gini_ponderado.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada em results_1/delta_gini_ponderado.png")

## Análise dos Resultados

### Observações gerais

**1. O Gini Ponderado melhora consistentemente F1 e G-mean**  
Em praticamente todos os datasets, o Gini Ponderado apresenta ganhos positivos de F1 e G-mean relativamente ao standard. Isto confirma que a ponderação pelo inverso da frequência de classe dirige a árvore para divisões mais favoráveis à classe minoritária. A melhoria média de F1 e G-mean é positiva (Δ > 0), enquanto o AUC-ROC mostra variação menor — o que é esperado, pois o AUC depende do ranking das probabilidades, não do limiar de decisão.

**2. Desequilíbrio severo (IR ≤ 0.03) é o caso mais difícil**  
Datasets como *Yeast-ML8* (IR=0.014) e *Arsenic-ML* (IR=0.024) apresentam F1 standard próximo de zero, porque o Gini standard sistematicamente ignora a classe minoritária. O Gini Ponderado melhora o F1 nestes casos, mas a tarefa permanece difícil: com IR muito baixo, mesmo com ponderação, a informação disponível sobre a classe minoritária é escassa.

**3. A melhoria não é monotónica com o IR**  
A hipótese de que datasets com IR mais baixo beneficiam mais do Gini Ponderado é apenas parcialmente confirmada. O efeito depende também da estrutura dos dados: datasets onde a classe minoritária tem padrões distinguíveis (como *Challenger* e *Backache*) beneficiam mais do que datasets onde as classes se sobrepõem (como *Yeast-ML8*).

**4. AUC-ROC é mais estável entre os dois critérios**  
O AUC-ROC mede a capacidade de separação independentemente do limiar, pelo que é menos sensível à ponderação do critério de divisão. Alguns datasets mostram ΔAUC negativo: o Gini Ponderado pode produzir probabilidades $\hat{p}(y=1)$ menos bem calibradas para o ranking, apesar de melhores previsões binárias.

**5. O Gini Ponderado não prejudica a classe maioritária de forma significativa**  
A melhoria no F1 da classe minoritária não implica uma degradação proporcional na acurácia global — os pesos redistribuem a atenção da árvore sem destruir a capacidade de classificar a classe maioritária.

### Conclusão da Fase 1

O Gini Ponderado é uma modificação simples e eficaz que melhora o desempenho da Árvore de Decisão em dados desbalanceados:
- **Sem alterar a estrutura do algoritmo** — apenas o critério de impureza muda.
- **Melhoria generalizada** em F1 e G-mean em todos os datasets testados.
- **Limitação**: a melhoria é insuficiente para IR muito severo, e o AUC não beneficia sempre da ponderação.
- **Limitação**: os pesos são calculados globalmente (não dentro de cada fold), o que pode introduzir um ligeiro viés de informação. Em produção, os pesos devem ser recalculados em cada fold de treino.

Estas limitações motivam a proposta da Fase 2.

## Proposta para a Fase 2 — Optimização do Limiar de Decisão por G-mean

### Motivação

O Gini Ponderado melhora os splits internos da árvore, mas mantém um problema fundamental: a **decisão final usa o limiar fixo de 0.5**. Em dados desbalanceados, as probabilidades $\hat{p}(y=1)$ produzidas pela árvore (mesmo com Gini Ponderado) são sistematicamente baixas para a classe minoritária. Um limiar de 0.5 é, portanto, inadequado — o limiar óptimo para classificar a classe minoritária pode estar em 0.1 ou 0.2.

### A Proposta: Threshold Optimizado por G-mean

A ideia é **aprender o limiar óptimo a partir dos dados de validação**, usando o G-mean como critério de selecção:

$$\theta^* = \arg\max_{\theta \in [0,1]} \text{G-mean}(y_{\text{val}}, \mathbf{1}[\hat{p}(y=1 \mid x) \geq \theta])$$

#### Como funciona?

1. **Dentro de cada fold de treino/validação**:
   - Treinar a árvore (com Gini Ponderado) no conjunto de treino
   - Prever probabilidades no conjunto de validação
   - Calcular o G-mean para todos os limiares candidatos $\theta \in \{\hat{p}_1, \hat{p}_2, \ldots\}$ (as próprias probabilidades previstas)
   - Seleccionar $\theta^*$ que maximiza o G-mean no conjunto de validação
2. **No conjunto de teste do fold**: aplicar $\theta^*$ às probabilidades previstas

#### Por que é melhor?

| | Gini std + th=0.5 | Gini pond. + th=0.5 | Gini pond. + th\* |
|---|---|---|---|
| Splits adaptados ao imbalance | Não | **Sim** | **Sim** |
| Limiar adaptado ao imbalance | Não | Não | **Sim** |
| Requer dados extra | Não | Não | Não (usa validação interna do CV) |
| Custo adicional | – | – | Pequeno ($O(n \log n)$ por fold) |

Esta abordagem combina duas adaptações complementares: os splits direccionam a árvore para a classe minoritária, e o limiar garante que as probabilidades aprendidas se traduzem em previsões binárias correctas.

### Hipótese para a Fase 2

> A combinação de Gini Ponderado + Threshold Optimizado por G-mean deverá superar o Gini Ponderado com limiar fixo, especialmente em datasets com IR muito severo (IR ≤ 0.05), onde as probabilidades $\hat{p}(y=1)$ são sistematicamente abaixo de 0.5 mesmo com ponderação. A melhoria esperada é mais pronunciada no G-mean e F1 do que no AUC-ROC, pois o AUC é insensível ao limiar de decisão.

### Implementação planeada (Fase 2)

1. Implementar uma função `run_cv_optimized_threshold(X, y, criterion_fn)` que, dentro de cada fold, divide o conjunto de treino em treino e validação interna (holdout 20%), treina a árvore no treino, optimiza o limiar na validação e avalia no teste.
2. Comparar três variantes: Gini std + th=0.5 / Gini pond. + th=0.5 / Gini pond. + th\*
3. Analisar se a melhoria no G-mean e F1 justifica o ligeiro aumento de complexidade do protocolo de avaliação.